# Análise dos resultados - TrashNet

Este notebook consolida os seis experimentos MLP/CNN. As curvas, hiperparâmetros e medições de custo vêm de `historicos/`; as métricas de teste são calculadas aqui com os modelos salvos e gravadas em `resultados/`. A seleção do modelo para a análise de erros usa exclusivamente o maior F1 macro de validação registrado no histórico. O F1 macro, média simples dos F1 das seis classes, é a métrica principal de seleção. O conjunto de teste é usado para avaliação final, não para seleção.

As métricas por classe são calculadas para todas as categorias na ordem `glass`, `paper`, `cardboard`, `plastic`, `metal`, `trash`, com `zero_division=0`: uma métrica indefinida por ausência de predições positivas é reportada como zero. O suporte é o número de imagens avaliadas por classe. As matrizes mostram classes reais nas linhas e previstas nas colunas, com contagens e percentuais por linha.

A célula seguinte gera os elementos numerados, as tabelas e a discussão interpretativa. A medição de tempo é a registrada nos históricos: relógio de parede por época de `model.fit` (treino e validação), sem download nem leitura/redimensionamento inicial das imagens. A limitação de suporte reduzido de `trash` no teste é considerada na interpretação.

In [1]:
import csv
import html
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from IPython.display import HTML, Markdown, display
from PIL import Image
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import common

DATA_DIR = ROOT / "dataset-resized"
HISTORY_DIR = ROOT / "historicos"
MODEL_DIR = ROOT / "modelos"
RESULT_DIR = ROOT / "resultados"
FIGURE_DIR = RESULT_DIR / "figuras"
RESULT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
CLASSES = common.CLASSES
EXPERIMENTS = [
    "mlp_128_sem",
    "mlp_128_com",
    "cnn_128_sem",
    "cnn_128_com",
    "cnn_256_sem",
    "cnn_512_sem",
]

with (ROOT / "split.json").open(encoding="utf-8") as file:
    split = json.load(file)
if split["classes"] != CLASSES:
    raise ValueError(f"Ordem inesperada das classes no split: {split['classes']}")

history_data = {
    name: json.loads((HISTORY_DIR / f"{name}.json").read_text(encoding="utf-8"))
    for name in EXPERIMENTS
}
distribution = []
for class_name in CLASSES:
    row = {"classe": class_name}
    for subset in ("train", "val", "test"):
        row[subset] = sum(item["label"] == class_name for item in split[subset])
    distribution.append(row)

test_paths, y_test = common.caminhos_e_rotulos(split, "test", str(DATA_DIR))
test_support = np.bincount(y_test, minlength=len(CLASSES))
predictions = {}
probabilities = {}
test_metrics = {}
confusion_matrices = {}
per_class_rows = []

for experiment in EXPERIMENTS:
    meta = history_data[experiment]["meta"]
    model_path = MODEL_DIR / f"{experiment}.keras"
    if not model_path.exists():
        raise FileNotFoundError(f"Modelo salvo ausente: {model_path}")
    model = tf.keras.models.load_model(model_path, compile=False)
    test_ds = common.criar_dataset(
        split, "test", meta["res"], meta["batch_size"], data_dir=str(DATA_DIR)
    )
    probs = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test,
        y_pred,
        labels=np.arange(len(CLASSES)),
        average=None,
        zero_division=0,
    )
    matrix = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASSES)))
    predictions[experiment] = y_pred
    probabilities[experiment] = probs
    confusion_matrices[experiment] = matrix
    test_metrics[experiment] = {
        "accuracy": float(np.mean(y_test == y_pred)),
        "f1_macro": float(np.mean(f1)),
        "classes": {
            class_name: {
                "precision": float(precision[index]),
                "recall": float(recall[index]),
                "f1": float(f1[index]),
                "support": int(support[index]),
            }
            for index, class_name in enumerate(CLASSES)
        },
        "confusion_matrix": matrix.astype(int).tolist(),
    }
    for index, class_name in enumerate(CLASSES):
        per_class_rows.append(
            {
                "experimento": experiment,
                "classe": class_name,
                "precision": float(precision[index]),
                "recall": float(recall[index]),
                "f1": float(f1[index]),
                "support": int(support[index]),
            }
        )
    del model, test_ds
    tf.keras.backend.clear_session()

selected_experiment = max(
    EXPERIMENTS, key=lambda name: history_data[name]["meta"]["val_f1_macro"]
)
selected_probs = probabilities[selected_experiment]
selected_pred = predictions[selected_experiment]
error_indices = np.flatnonzero(y_test != selected_pred)
total_error_count = int(len(error_indices))
error_unit = "imagem" if total_error_count == 1 else "imagens"
trash_support = int(test_support[CLASSES.index("trash")])
error_rows = []
for index in error_indices:
    predicted_class = int(selected_pred[index])
    error_rows.append(
        {
            "path": str(Path(test_paths[index]).relative_to(ROOT)),
            "real": CLASSES[int(y_test[index])],
            "prevista": CLASSES[predicted_class],
            "probabilidade_prevista": float(selected_probs[index, predicted_class]),
        }
    )
error_rows = error_rows[:6]

comparison_rows = []
hardware_rows = []
for experiment in EXPERIMENTS:
    meta = history_data[experiment]["meta"]
    metrics = test_metrics[experiment]
    comparison_rows.append(
        {
            "experimento": experiment,
            "acuracia_teste": metrics["accuracy"],
            "f1_macro_teste": metrics["f1_macro"],
            "acuracia_validacao": meta["val_accuracy"],
            "f1_macro_validacao": meta["val_f1_macro"],
            "parametros_treinaveis": meta["parametros_treinaveis"],
            "parametros_totais": meta["parametros_totais"],
            "modelo_mb": meta["tamanho_modelo_salvo_mb"],
            "epocas": meta["epocas_rodadas"],
            "epoca_modelo": meta["epoca_do_modelo_salvo"],
            "tempo_total_s": meta["tempo_total_treino_s"],
            "tempo_medio_epoca_s": meta["tempo_medio_por_epoca_s"],
            "batch_size": meta["batch_size"],
            "gpu": ", ".join(meta["hardware"]["gpu"] or []) or "não informada",
            "cpu": meta["hardware"]["cpu"],
            "ram_pico_mb": meta.get("memoria_ram_pico_mb"),
            "gpu_pico_mb": meta.get("memoria_gpu_pico_mb"),
        }
    )
    hardware_rows.append(
        {
            "experimento": experiment,
            "gpu": ", ".join(meta["hardware"]["gpu"] or []) or "não informada",
            "cpu": meta["hardware"]["cpu"],
            "tensorflow": meta["hardware"]["tensorflow"],
            "python": meta["hardware"]["python"],
            "ram_pico_mb": meta.get("memoria_ram_pico_mb"),
            "gpu_pico_mb": meta.get("memoria_gpu_pico_mb"),
        }
    )

results_payload = {
    "classes": CLASSES,
    "support_teste": {
        class_name: int(test_support[index])
        for index, class_name in enumerate(CLASSES)
    },
    "distribuicao": distribution,
    "experimentos": test_metrics,
    "modelo_selecionado_por_f1_macro_validacao": selected_experiment,
    "total_erros_modelo_selecionado": total_error_count,
    "erros_modelo_selecionado": error_rows,
}
(RESULT_DIR / "metricas_teste.json").write_text(
    json.dumps(results_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)

for filename, rows in (
    ("distribuicao_por_conjunto.csv", distribution),
    ("comparacao_experimentos.csv", comparison_rows),
    ("metricas_por_classe.csv", per_class_rows),
    ("erros_modelo_selecionado.csv", error_rows),
):
    if rows:
        with (RESULT_DIR / filename).open("w", newline="", encoding="utf-8-sig") as file:
            writer = csv.DictWriter(file, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


def show_table(caption, rows, columns):
    display(Markdown(caption))
    header = "".join(f"<th>{html.escape(label)}</th>" for label, _ in columns)
    body = []
    for row in rows:
        cells = []
        for _, key in columns:
            value = row.get(key)
            if isinstance(value, float):
                value = f"{value:.3f}"
            elif value is None:
                value = "n/d"
            cells.append(f"<td>{html.escape(str(value))}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(
        HTML(
            "<table><thead><tr>"
            + header
            + "</tr></thead><tbody>"
            + "".join(body)
            + "</tbody></table>"
        )
    )


def save_and_show_figure(fig, filename, caption):
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=160, bbox_inches="tight")
    display(Markdown(caption))
    display(fig)
    plt.close(fig)


show_table(
    "**Tabela 1 — Distribuição das imagens por classe e subconjunto.**",
    distribution,
    [("Classe", "classe"), ("Treino", "train"), ("Validação", "val"), ("Teste", "test")],
)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CLASSES))
width = 0.24
for offset, subset, label in (
    (-width, "train", "Treino"),
    (0, "val", "Validação"),
    (width, "test", "Teste"),
):
    values = [row[subset] for row in distribution]
    bars = ax.bar(x + offset, values, width, label=label)
    ax.bar_label(bars, padding=2, fontsize=8)
ax.set_xticks(x, CLASSES)
ax.set_ylabel("Quantidade de imagens")
ax.set_title("Distribuição das classes por subconjunto")
ax.legend()
save_and_show_figure(
    fig,
    "figura_01_distribuicao.png",
    "**Figura 1 — Quantidade de imagens por classe em treino, validação e teste.** "
    "A distribuição é desbalanceada; `trash` tem o menor suporte.",
)

examples = {}
for item in split["train"]:
    examples.setdefault(item["label"], [])
    if len(examples[item["label"]]) < 2:
        examples[item["label"]].append(item["path"])
if any(len(examples.get(class_name, [])) < 2 for class_name in CLASSES):
    raise ValueError("Não há duas imagens de treino disponíveis para cada classe.")
fig, axes = plt.subplots(len(CLASSES), 2, figsize=(7, 17))
for row, class_name in enumerate(CLASSES):
    for column, relative_path in enumerate(examples[class_name]):
        with Image.open(DATA_DIR / relative_path) as image:
            axes[row, column].imshow(image.convert("RGB"))
        axes[row, column].set_title(f"{class_name} — {Path(relative_path).name}")
        axes[row, column].axis("off")
save_and_show_figure(
    fig,
    "figura_02_exemplos_por_classe.png",
    "**Figura 2 — Exemplos de treino: duas imagens por classe.**",
)

same_image = next(item for item in split["train"] if item["label"] == "metal")
with Image.open(DATA_DIR / same_image["path"]) as original:
    source_image = original.convert("RGB")
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, resolution in zip(axes, (128, 256, 512)):
    display_size = (resolution, resolution * 3 // 4)
    ax.imshow(source_image.resize(display_size, Image.Resampling.LANCZOS))
    ax.set_title(f"{resolution} × {resolution * 3 // 4}")
    ax.axis("off")
save_and_show_figure(
    fig,
    "figura_03_mesma_imagem_resolucoes.png",
    f"**Figura 3 — A mesma imagem ({Path(same_image['path']).name}) nas três resoluções dos experimentos CNN.**",
)

fig, axes = plt.subplots(3, 2, figsize=(14, 14), sharex=False)
for ax, experiment in zip(axes.flat, EXPERIMENTS):
    record = history_data[experiment]
    history = record["historico"]
    epochs = np.arange(1, len(history["loss"]) + 1)
    ax.plot(epochs, history["loss"], label="Treino")
    ax.plot(epochs, history["val_loss"], label="Validação")
    ax.axvline(record["meta"]["epoca_do_modelo_salvo"], color="gray", linestyle="--", linewidth=1)
    ax.set_title(experiment)
    ax.set_xlabel("Época")
    ax.set_ylabel("Perda")
    ax.legend(fontsize=8)
save_and_show_figure(
    fig,
    "figura_04_curvas_perda.png",
    "**Figura 4 — Perda de treino e validação por época nos seis experimentos.** "
    "A linha tracejada marca a época do modelo salvo.",
)

fig, axes = plt.subplots(3, 2, figsize=(14, 14), sharex=False)
for ax, experiment in zip(axes.flat, EXPERIMENTS):
    record = history_data[experiment]
    history = record["historico"]
    epochs = np.arange(1, len(history["accuracy"]) + 1)
    ax.plot(epochs, history["accuracy"], label="Treino")
    ax.plot(epochs, history["val_accuracy"], label="Validação")
    ax.axvline(record["meta"]["epoca_do_modelo_salvo"], color="gray", linestyle="--", linewidth=1)
    ax.set_title(experiment)
    ax.set_xlabel("Época")
    ax.set_ylabel("Acurácia")
    ax.legend(fontsize=8)
save_and_show_figure(
    fig,
    "figura_05_curvas_acuracia.png",
    "**Figura 5 — Acurácia de treino e validação por época nos seis experimentos.** "
    "A distância entre as curvas ajuda a identificar sinais de sobreajuste.",
)

show_table(
    "**Tabela 2 — Métricas comparativas dos seis experimentos.** "
    "O F1 macro de validação é o critério de seleção.",
    comparison_rows,
    [
        ("Experimento", "experimento"),
        ("Acurácia teste", "acuracia_teste"),
        ("F1 macro teste", "f1_macro_teste"),
        ("Acurácia validação", "acuracia_validacao"),
        ("F1 macro validação", "f1_macro_validacao"),
    ],
)
show_table(
    "**Tabela 3 — Parâmetros, tamanho salvo e custo computacional.** "
    "Parâmetros, tamanho, épocas e custos foram copiados dos históricos.",
    comparison_rows,
    [
        ("Experimento", "experimento"),
        ("Parâmetros treináveis", "parametros_treinaveis"),
        ("Parâmetros totais", "parametros_totais"),
        ("Modelo (MB)", "modelo_mb"),
        ("Épocas treinadas", "epocas"),
        ("Época do modelo", "epoca_modelo"),
        ("Tempo total (s)", "tempo_total_s"),
        ("Média/época (s)", "tempo_medio_epoca_s"),
        ("Lote", "batch_size"),
    ],
)
show_table(
    "**Tabela 4 — Hardware e memória registrada por experimento.** "
    "`n/d` indica que o histórico não registrou a medição.",
    hardware_rows,
    [
        ("Experimento", "experimento"),
        ("GPU", "gpu"),
        ("CPU", "cpu"),
        ("TensorFlow", "tensorflow"),
        ("Python", "python"),
        ("RAM pico (MB)", "ram_pico_mb"),
        ("GPU pico (MB)", "gpu_pico_mb"),
    ],
)
show_table(
    "**Tabela 5 — Precisão, recall, F1 e suporte por classe no teste.** "
    "As métricas foram calculadas para todas as classes com `zero_division=0`.",
    per_class_rows,
    [
        ("Experimento", "experimento"),
        ("Classe", "classe"),
        ("Precisão", "precision"),
        ("Recall", "recall"),
        ("F1", "f1"),
        ("Suporte", "support"),
    ],
)
display(
    Markdown(
        "**Método e hardware.** O tempo total e o tempo por época seguem `time.time()` "
        "registrado em cada época de `model.fit`, incluindo treino e validação; download e "
        "leitura/redimensionamento inicial são excluídos. O lote é informado na Tabela 3; "
        "CPU, GPU, versões de TensorFlow/Python e picos de RAM/GPU estão na Tabela 4. "
        "A ausência de valores de pico é marcada como `n/d`."
    )
)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, experiment in zip(axes.flat, EXPERIMENTS):
    matrix = confusion_matrices[experiment]
    row_totals = matrix.sum(axis=1, keepdims=True)
    row_percent = np.divide(
        matrix * 100,
        row_totals,
        out=np.zeros_like(matrix, dtype=float),
        where=row_totals != 0,
    )
    ax.imshow(row_percent, cmap="Blues", vmin=0, vmax=100)
    ax.set_title(experiment)
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("Classe prevista")
    ax.set_ylabel("Classe real")
    for row in range(len(CLASSES)):
        for column in range(len(CLASSES)):
            ax.text(
                column,
                row,
                f"{matrix[row, column]}\n{row_percent[row, column]:.1f}%",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if row_percent[row, column] > 55 else "black",
            )
save_and_show_figure(
    fig,
    "figura_06_matrizes_confusao.png",
    "**Figura 6 — Matrizes de confusão no teste.** Cada célula contém contagem e "
    "percentual da classe real correspondente (linha); classes reais estão nas linhas e "
    "classes previstas nas colunas, na ordem fixa declarada no início.",
)

resolution_experiments = ["cnn_128_sem", "cnn_256_sem", "cnn_512_sem"]
resolution_rows = [next(row for row in comparison_rows if row["experimento"] == name) for name in resolution_experiments]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
resolutions = [128, 256, 512]
axes[0].plot(
    resolutions,
    [row["acuracia_teste"] for row in resolution_rows],
    marker="o",
    label="Acurácia",
)
axes[0].plot(
    resolutions,
    [row["f1_macro_teste"] for row in resolution_rows],
    marker="o",
    label="F1 macro",
)
axes[0].set_xlabel("Resolução (largura)")
axes[0].set_ylabel("Métrica no teste")
axes[0].set_xticks(resolutions)
axes[0].legend()
axes[0].set_title("Desempenho no teste")
axes[1].plot(
    resolutions,
    [row["tempo_medio_epoca_s"] for row in resolution_rows],
    marker="o",
    color="tab:orange",
)
axes[1].set_xlabel("Resolução (largura)")
axes[1].set_ylabel("Segundos por época")
axes[1].set_xticks(resolutions)
axes[1].set_title("Tempo médio por época")
axes[2].plot(
    resolutions,
    [row["tempo_total_s"] for row in resolution_rows],
    marker="o",
    color="tab:green",
)
axes[2].set_xlabel("Resolução (largura)")
axes[2].set_ylabel("Segundos de treinamento")
axes[2].set_xticks(resolutions)
axes[2].set_title("Tempo total")
save_and_show_figure(
    fig,
    "figura_07_efeito_resolucao.png",
    "**Figura 7 — Efeito da resolução na CNN sem pesos de classe: métricas de teste, "
    "tempo médio por época e tempo total.**",
)

display(
    Markdown(
        f"**Seleção para análise de erros:** `{selected_experiment}`, por apresentar o "
        f"maior F1 macro de validação registrado em `historicos/` "
        f"({history_data[selected_experiment]['meta']['val_f1_macro']:.3f}). "
        f"Ele classificou incorretamente {total_error_count} {error_unit} no teste; "
        f"a tabela e a figura apresentam até seis. "
        f"A classe `trash` tem suporte de {trash_support} imagens no teste; "
        f"cada erro equivale a {100 / trash_support:.1f} pontos percentuais de recall. "
        "Esse suporte pequeno significa que cada erro altera de forma importante "
        "as métricas dessa classe, portanto os resultados devem ser interpretados com cautela."
    )
)
if error_rows:
    show_table(
        "**Tabela 7 — Até seis erros do modelo selecionado no teste.** "
        "A probabilidade é a atribuída à classe prevista.",
        error_rows,
        [
            ("Imagem", "path"),
            ("Classe real", "real"),
            ("Classe prevista", "prevista"),
            ("Probabilidade prevista", "probabilidade_prevista"),
        ],
    )
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, error in zip(axes.flat, error_rows):
        with Image.open(ROOT / error["path"]) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(
            f"Real: {error['real']}\nPrevista: {error['prevista']} "
            f"({error['probabilidade_prevista']:.3f})"
        )
        ax.axis("off")
    for ax in axes.flat[len(error_rows) :]:
        ax.axis("off")
    save_and_show_figure(
        fig,
        "figura_08_erros_teste.png",
        "**Figura 8 — Imagens classificadas incorretamente pelo modelo selecionado.**",
    )
else:
    display(Markdown("**Análise de erros:** o modelo selecionado não errou nenhuma imagem do teste."))

selected_matrix = confusion_matrices[selected_experiment].copy()
np.fill_diagonal(selected_matrix, 0)
confusion_pairs = sorted(
    (
        (int(selected_matrix[row, column]), CLASSES[row], CLASSES[column])
        for row in range(len(CLASSES))
        for column in range(len(CLASSES))
    ),
    reverse=True,
)
top_confusions = [item for item in confusion_pairs if item[0] > 0][:3]
if top_confusions:
    confusion_text = "; ".join(
        f"{actual}→{predicted}: {count}" for count, actual, predicted in top_confusions
    )
else:
    confusion_text = "não houve confusões entre classes"
display(
    Markdown(
        f"**Interpretação das confusões:** no modelo selecionado, os pares com maior "
        f"contagem fora da diagonal são {confusion_text}. Consulte a Figura 6 para a "
        "distribuição completa de erros em cada experimento."
    )
)

for plain, weighted, model_label in (
    ("mlp_128_sem", "mlp_128_com", "MLP"),
    ("cnn_128_sem", "cnn_128_com", "CNN"),
):
    plain_trash = test_metrics[plain]["classes"]["trash"]
    weighted_trash = test_metrics[weighted]["classes"]["trash"]
    delta_precision = weighted_trash["precision"] - plain_trash["precision"]
    delta_recall = weighted_trash["recall"] - plain_trash["recall"]
    delta_f1 = weighted_trash["f1"] - plain_trash["f1"]
    direction_precision = "aumentou" if delta_precision > 0 else "diminuiu" if delta_precision < 0 else "não mudou"
    direction_recall = "aumentou" if delta_recall > 0 else "diminuiu" if delta_recall < 0 else "não mudou"
    direction_f1 = "aumentou" if delta_f1 > 0 else "diminuiu" if delta_f1 < 0 else "não mudou"
    display(
        Markdown(
            f"**Efeito dos pesos em `trash` ({model_label}):** em comparação ao modelo "
            f"sem pesos, a precisão {direction_precision} ({delta_precision:+.3f}), "
            f"o recall {direction_recall} ({delta_recall:+.3f}) e o F1 "
            f"{direction_f1} ({delta_f1:+.3f}) no teste. O recall deve ser lido junto "
            "da precisão: um ganho de cobertura pode vir acompanhado de mais falsos positivos."
        )
    )

curve_diagnostics = []
for experiment in EXPERIMENTS:
    record = history_data[experiment]
    curve = record["historico"]
    epoch_index = record["meta"]["epoca_do_modelo_salvo"] - 1
    curve_diagnostics.append(
        {
            "experimento": experiment,
            "epoca_modelo": record["meta"]["epoca_do_modelo_salvo"],
            "acuracia_treino": curve["accuracy"][epoch_index],
            "acuracia_validacao": curve["val_accuracy"][epoch_index],
            "gap_acuracia": curve["accuracy"][epoch_index] - curve["val_accuracy"][epoch_index],
            "gap_perda": curve["val_loss"][epoch_index] - curve["loss"][epoch_index],
        }
    )
largest_gap = max(curve_diagnostics, key=lambda row: abs(row["gap_acuracia"]))
gap_direction = (
    "treino acima da validação"
    if largest_gap["gap_acuracia"] > 0
    else "validação acima do treino"
    if largest_gap["gap_acuracia"] < 0
    else "treino e validação iguais"
)
show_table(
    "**Tabela 6 — Diferença entre treino e validação na época do modelo salvo.** "
    "Gap de acurácia = treino menos validação; gap de perda = validação menos treino.",
    curve_diagnostics,
    [
        ("Experimento", "experimento"),
        ("Época", "epoca_modelo"),
        ("Acurácia treino", "acuracia_treino"),
        ("Acurácia validação", "acuracia_validacao"),
        ("Gap acurácia", "gap_acuracia"),
        ("Gap perda", "gap_perda"),
    ],
)
display(
    Markdown(
        f"**Curvas e sobreajuste:** o maior gap de acurácia na época salva é "
        f"{abs(largest_gap['gap_acuracia']):.3f} em `{largest_gap['experimento']}` "
        f"({gap_direction}). As perdas de treino e validação se aproximam nas épocas "
        "finais, sem divergência consistente típica de sobreajuste; as métricas de "
        "validação oscilam mais no início, então a estabilidade ainda merece atenção."
    )
)

resolution_128 = resolution_rows[0]
resolution_256 = resolution_rows[1]
resolution_512 = resolution_rows[-1]
best_resolution_row = max(resolution_rows, key=lambda row: row["f1_macro_teste"])
best_resolution = history_data[best_resolution_row["experimento"]]["meta"]["res"]
delta_f1_256 = resolution_256["f1_macro_teste"] - resolution_128["f1_macro_teste"]
delta_f1_512 = resolution_512["f1_macro_teste"] - resolution_128["f1_macro_teste"]
time_ratio_256 = resolution_256["tempo_total_s"] / resolution_128["tempo_total_s"]
time_ratio_512 = resolution_512["tempo_total_s"] / resolution_128["tempo_total_s"]
params_128 = resolution_128["parametros_totais"]
params_256 = resolution_256["parametros_totais"]
display(
    Markdown(
        f"**Resolução versus custo:** a maior F1 macro entre estas CNNs foi em "
        f"{best_resolution} pixels de largura. Contra a CNN sem pesos em resolução menor, a "
        f"resolução intermediária alterou o F1 macro em {delta_f1_256:+.3f} e custou "
        f"{time_ratio_256:.2f} vezes mais tempo total; a resolução maior alterou o F1 "
        f"macro em {delta_f1_512:+.3f} e custou {time_ratio_512:.2f} vezes mais. "
        "Portanto, o maior tamanho não compensou automaticamente seu custo. Além disso, "
        f"os históricos registram {params_128} parâmetros totais na CNN de resolução menor "
        f"e {params_256} nas outras resoluções: a diferença de arquitetura/parâmetros "
        "limita atribuir todo o efeito apenas à resolução."
    )
)

zero_prediction_classes = []
for experiment in EXPERIMENTS:
    predicted_support = np.bincount(predictions[experiment], minlength=len(CLASSES))
    for index, class_name in enumerate(CLASSES):
        if predicted_support[index] == 0:
            zero_prediction_classes.append(f"{experiment}: {class_name}")
if zero_prediction_classes:
    display(
        Markdown(
            "**Divisão por zero:** `zero_division=0` definiu como zero as métricas "
            "indefinidas para classes sem predições positivas: "
            + ", ".join(zero_prediction_classes)
            + "."
        )
    )
else:
    display(
        Markdown(
            "**Divisão por zero:** não houve classe sem predições positivas nos relatórios "
            "de teste; ainda assim, `zero_division=0` foi aplicado para garantir tratamento "
            "definido em qualquer divisão por zero."
        )
    )

print(f"Resultados numéricos e tabelas salvos em: {RESULT_DIR}")
print(f"Figuras salvas em: {FIGURE_DIR}")


ModuleNotFoundError: No module named 'tensorflow.python'